# Brazilian E-Commerce Business Analytics — Olist
**Proyek Akhir Analisis Data Dicoding**

Analisis performa penjualan, segmentasi pelanggan RFM, dan performa pengiriman pada Brazilian E-Commerce Public Dataset by Olist.

> Dataset asli terdiri dari beberapa tabel CSV relasional. Letakkan file CSV Olist di folder `data/` sebelum menjalankan notebook ini. Notebook akan memeriksa file, membersihkan data, melakukan EDA, membuat visualisasi, dan mengekspor data untuk dashboard Streamlit.

## Pertanyaan bisnis (SMART)
1. Bagaimana perkembangan pendapatan dan jumlah pesanan setiap bulan sepanjang periode transaksi yang tersedia, dan bulan mana yang memiliki performa tertinggi?
2. Bagaimana karakteristik pelanggan berdasarkan Recency, Frequency, dan Monetary (RFM), serta segmen mana yang perlu diprioritaskan untuk retensi?
3. Bagaimana tingkat keterlambatan dan rata-rata waktu pengiriman sepanjang periode transaksi yang tersedia, serta apa implikasinya bagi perbaikan logistik?

**Catatan periode:** periode analisis ditentukan secara dinamis berdasarkan tanggal transaksi valid dalam dataset, bukan diasumsikan sebagai rentang tetap.

## 1. Import library dan konfigurasi

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid", palette="deep")

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "dashboard"
OUTPUT_DIR.mkdir(exist_ok=True)

required_files = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
}

## 2. Gathering Data

In [ ]:
def load_csv(filename, parse_dates=None):
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(
            f"File {filename} belum ditemukan di {DATA_DIR}. "
            "Unduh Brazilian E-Commerce Public Dataset by Olist dari Kaggle "
            "atau Google Drive Dicoding, lalu ekstrak seluruh CSV ke folder data/."
        )
    return pd.read_csv(path, parse_dates=parse_dates)

orders = load_csv(
    required_files["orders"],
    parse_dates=[
        "order_purchase_timestamp", "order_approved_at",
        "order_delivered_carrier_date", "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)
customers = load_csv(required_files["customers"])
items = load_csv(required_files["items"])
payments = load_csv(required_files["payments"])

print("Ukuran data:")
for name, frame in {
    "orders": orders, "customers": customers,
    "items": items, "payments": payments
}.items():
    print(f"{name:12s}: {frame.shape[0]:,} baris × {frame.shape[1]} kolom")
display(orders.head())

### Ringkasan Gathering Data
Empat tabel utama dimuat untuk menjawab pertanyaan bisnis: `orders` sebagai sumber waktu dan status pesanan, `customers` untuk identitas pelanggan unik, `items` untuk nilai transaksi, serta `payments` sebagai informasi pembayaran. Relasi utama dilakukan melalui `order_id` dan `customer_id`.

## 3. Assessing Data

In [ ]:
def data_quality_report(df, name):
    report = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2),
        "duplicate_count": [df.duplicated().sum()] + [0] * (len(df.columns)-1),
        "dtype": df.dtypes.astype(str)
    })
    print(f"\n{name}: {df.shape}")
    display(report[report["missing_count"].gt(0) | report["duplicate_count"].gt(0)])
    return report

quality_orders = data_quality_report(orders, "orders")
quality_customers = data_quality_report(customers, "customers")
quality_items = data_quality_report(items, "items")
quality_payments = data_quality_report(payments, "payments")

print("Duplikasi primary key:")
print("orders order_id:", orders["order_id"].duplicated().sum())
print("customers customer_id:", customers["customer_id"].duplicated().sum())
print("items duplicate rows:", items.duplicated().sum())
print("payments duplicate rows:", payments.duplicated().sum())

print("\nRentang tanggal transaksi:")
print(orders["order_purchase_timestamp"].min(), "s.d.", orders["order_purchase_timestamp"].max())
print("\nStatus pesanan:")
display(orders["order_status"].value_counts(dropna=False))

### Temuan awal dan rencana penanganan kualitas data
- **Missing value:** tanggal persetujuan/pengiriman dapat kosong pada pesanan yang belum selesai atau dibatalkan. Nilai tersebut tidak diisi dengan tanggal buatan; analisis durasi pengiriman hanya memakai pesanan dengan tanggal pengiriman valid.
- **Duplikasi:** baris duplikat identik pada tabel transaksi diperiksa dan dihapus. Kunci `order_id` pada tabel pesanan dan `customer_id` pada tabel pelanggan harus unik.
- **Invalid value:** nilai harga negatif atau nol dan timestamp yang tidak logis diperiksa. Baris yang tidak valid dikeluarkan dari metrik yang membutuhkan nilai tersebut, bukan diganti sembarangan.
- **Inconsistent value:** kolom tanggal dikonversi ke tipe datetime dan status dipertahankan sesuai kategori sumber.

## 4. Cleaning Data

In [ ]:
# Salin data mentah agar tidak mengubah objek awal
orders_clean = orders.copy()
customers_clean = customers.copy()
items_clean = items.copy()
payments_clean = payments.copy()

# Hapus duplikasi identik
orders_clean = orders_clean.drop_duplicates()
customers_clean = customers_clean.drop_duplicates()
items_clean = items_clean.drop_duplicates()
payments_clean = payments_clean.drop_duplicates()

# Pastikan primary key tabel utama unik
orders_clean = orders_clean.drop_duplicates(subset="order_id", keep="first")
customers_clean = customers_clean.drop_duplicates(subset="customer_id", keep="first")

# Standarisasi tipe data tanggal
date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in date_cols:
    orders_clean[col] = pd.to_datetime(orders_clean[col], errors="coerce")

# Pastikan kolom numerik valid
items_clean["price"] = pd.to_numeric(items_clean["price"], errors="coerce")
items_clean["freight_value"] = pd.to_numeric(items_clean["freight_value"], errors="coerce")
payments_clean["payment_value"] = pd.to_numeric(payments_clean["payment_value"], errors="coerce")

# Item dengan harga tidak valid tidak digunakan untuk analisis revenue
items_clean = items_clean[
    items_clean["price"].notna() & (items_clean["price"] > 0)
].copy()
payments_clean = payments_clean[
    payments_clean["payment_value"].notna() & (payments_clean["payment_value"] >= 0)
].copy()

print("Ukuran setelah cleaning:")
for name, frame in {
    "orders_clean": orders_clean, "customers_clean": customers_clean,
    "items_clean": items_clean, "payments_clean": payments_clean
}.items():
    print(f"{name:16s}: {frame.shape}")

### Ringkasan Cleaning Data

In [ ]:
cleaning_summary = pd.DataFrame({
    "dataset": ["orders", "customers", "items", "payments"],
    "rows_before": [len(orders), len(customers), len(items), len(payments)],
    "rows_after": [len(orders_clean), len(customers_clean), len(items_clean), len(payments_clean)]
})
cleaning_summary["rows_removed"] = cleaning_summary["rows_before"] - cleaning_summary["rows_after"]
display(cleaning_summary)

## 5. Data Preparation dan Feature Engineering

In [ ]:
# Agregasi item per pesanan: satu order dapat memiliki beberapa item
order_revenue = (
    items_clean.groupby("order_id", as_index=False)
    .agg(revenue=("price", "sum"),
         freight_value=("freight_value", "sum"),
         item_count=("order_item_id", "count"))
)

# Agregasi pembayaran per pesanan
payment_agg = (
    payments_clean.groupby("order_id", as_index=False)
    .agg(payment_total=("payment_value", "sum"))
)

analysis = (
    orders_clean
    .merge(customers_clean[["customer_id", "customer_unique_id", "customer_city", "customer_state"]],
           on="customer_id", how="left")
    .merge(order_revenue, on="order_id", how="left")
    .merge(payment_agg, on="order_id", how="left")
)

analysis["order_month"] = analysis["order_purchase_timestamp"].dt.to_period("M").astype("string")
analysis["delivery_days"] = (
    analysis["order_delivered_customer_date"] - analysis["order_purchase_timestamp"]
).dt.total_seconds() / 86400
analysis["is_late"] = (
    analysis["order_delivered_customer_date"] > analysis["order_estimated_delivery_date"]
)
analysis["is_late"] = analysis["is_late"].where(
    analysis["order_delivered_customer_date"].notna() &
    analysis["order_estimated_delivery_date"].notna()
)
analysis["delivery_days"] = analysis["delivery_days"].where(analysis["delivery_days"] >= 0)

# Untuk analisis transaksi, gunakan pesanan dengan tanggal pembelian valid dan status selain canceled
valid_orders = analysis[
    analysis["order_purchase_timestamp"].notna() &
    analysis["order_status"].ne("canceled")
].copy()

period_start = valid_orders["order_purchase_timestamp"].min()
period_end = valid_orders["order_purchase_timestamp"].max()
print(f"Periode data valid: {period_start:%d %B %Y} – {period_end:%d %B %Y}")
print(f"Jumlah order valid: {valid_orders['order_id'].nunique():,}")

## 6. Exploratory Data Analysis (EDA)

### 6.1 Pertanyaan Bisnis 1 — Performa penjualan bulanan

In [ ]:
monthly_sales = (
    valid_orders.dropna(subset=["order_month"])
    .groupby("order_month", as_index=False)
    .agg(total_revenue=("revenue", "sum"),
         total_orders=("order_id", "nunique"),
         average_order_value=("revenue", "mean"))
)
monthly_sales["month_date"] = pd.to_datetime(monthly_sales["order_month"] + "-01")
monthly_sales = monthly_sales.sort_values("month_date")

display(monthly_sales.head())
best_month = monthly_sales.loc[monthly_sales["total_revenue"].idxmax()]
print("Bulan dengan pendapatan tertinggi:", best_month["order_month"])
print("Pendapatan tertinggi:", f"R$ {best_month['total_revenue']:,.2f}")
print("Total order pada bulan tersebut:", f"{best_month['total_orders']:,}")

**Interpretasi EDA:** agregasi dilakukan per bulan menggunakan tanggal pembelian. Pendapatan dihitung dari penjumlahan harga item (`price`) per pesanan; ongkos kirim tidak digabungkan ke revenue produk. Jumlah pesanan dihitung dengan `nunique(order_id)` agar pesanan multi-item tidak dihitung berulang.

### 6.2 Pertanyaan Bisnis 2 — Segmentasi pelanggan dengan RFM

In [ ]:
rfm_base = valid_orders.dropna(subset=["order_purchase_timestamp", "customer_unique_id"]).copy()
snapshot_date = rfm_base["order_purchase_timestamp"].max() + pd.Timedelta(days=1)

rfm = (
    rfm_base.groupby("customer_unique_id")
    .agg(
        last_purchase=("order_purchase_timestamp", "max"),
        frequency=("order_id", "nunique"),
        monetary=("revenue", "sum")
    )
    .reset_index()
)
rfm["recency"] = (snapshot_date - rfm["last_purchase"]).dt.days

# Skor kuartil: Recency semakin kecil semakin baik; Frequency/Monetary semakin besar semakin baik
rfm["R_score"] = pd.qcut(rfm["recency"].rank(method="first"), 4, labels=[4, 3, 2, 1]).astype(int)
rfm["F_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["M_score"] = pd.qcut(rfm["monetary"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["RFM_score"] = rfm[["R_score", "F_score", "M_score"]].sum(axis=1)

def segment_customer(row):
    if row["R_score"] >= 3 and row["F_score"] >= 3 and row["M_score"] >= 3:
        return "Champions"
    if row["R_score"] >= 3 and row["F_score"] >= 2:
        return "Loyal Customers"
    if row["R_score"] <= 2 and row["F_score"] >= 3:
        return "At Risk"
    if row["R_score"] <= 2 and row["F_score"] <= 2:
        return "Hibernating"
    return "Potential Loyalists"

rfm["segment"] = rfm.apply(segment_customer, axis=1)
rfm_summary = (
    rfm.groupby("segment", as_index=False)
    .agg(customers=("customer_unique_id", "nunique"),
         avg_recency=("recency", "mean"),
         avg_frequency=("frequency", "mean"),
         total_monetary=("monetary", "sum"))
    .sort_values("customers", ascending=False)
)
rfm_summary["customer_share_pct"] = rfm_summary["customers"] / rfm["customer_unique_id"].nunique() * 100
display(rfm_summary)

**Tujuan RFM:** mengelompokkan pelanggan berdasarkan kebaruan transaksi (Recency), jumlah transaksi (Frequency), dan total belanja (Monetary). Segmentasi ini merupakan rule-based/manual grouping berbasis skor kuartil, bukan model machine learning. Karena customer ID operasional Olist dapat berbeda untuk pembelian yang sama, identitas pelanggan menggunakan `customer_unique_id`.

### 6.3 Pertanyaan Bisnis 3 — Performa pengiriman

In [ ]:
delivery = valid_orders.dropna(
    subset=["order_delivered_customer_date", "order_estimated_delivery_date", "delivery_days"]
).copy()

delivery_summary = pd.DataFrame({
    "metric": ["total_delivered_orders", "late_orders", "late_rate_pct", "avg_delivery_days", "median_delivery_days"],
    "value": [
        delivery["order_id"].nunique(),
        delivery.loc[delivery["is_late"], "order_id"].nunique(),
        delivery["is_late"].mean() * 100,
        delivery["delivery_days"].mean(),
        delivery["delivery_days"].median()
    ]
})
display(delivery_summary)

delivery_by_month = (
    delivery.groupby("order_month", as_index=False)
    .agg(delivered_orders=("order_id", "nunique"),
         late_rate_pct=("is_late", lambda x: x.mean() * 100),
         avg_delivery_days=("delivery_days", "mean"))
    .sort_values("order_month")
)
display(delivery_by_month.head())

**Definisi keterlambatan:** pesanan dianggap terlambat jika tanggal aktual diterima pelanggan lebih besar daripada tanggal estimasi pengiriman. Pesanan tanpa kedua tanggal tersebut tidak dimasukkan ke perhitungan tingkat keterlambatan.

## 7. Visualization & Explanatory Analysis

### Visualisasi 1 — Tren pendapatan bulanan

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=monthly_sales, x="month_date", y="total_revenue", marker="o", ax=ax)
ax.set(title="Tren Pendapatan Bulanan Olist", xlabel="Bulan", ylabel="Pendapatan (BRL)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

### Visualisasi 2 — Jumlah pelanggan per segmen RFM

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=rfm_summary, x="segment", y="customers", ax=ax)
ax.set(title="Distribusi Segmen Pelanggan RFM", xlabel="Segmen Pelanggan", ylabel="Jumlah Pelanggan")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

### Visualisasi 3 — Tingkat keterlambatan pengiriman bulanan

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=delivery_by_month, x="order_month", y="late_rate_pct", marker="o", ax=ax)
ax.set(title="Tingkat Keterlambatan Pengiriman per Bulan", xlabel="Bulan", ylabel="Pesanan Terlambat (%)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

### Visualisasi 4 — Distribusi durasi pengiriman

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(data=delivery, x="delivery_days", bins=40, kde=True, ax=ax)
ax.set(title="Distribusi Durasi Pengiriman", xlabel="Hari sejak pembelian hingga diterima", ylabel="Jumlah Pesanan")
plt.tight_layout()
plt.show()

## 8. Conclusion & Recommendation

In [ ]:
# Kesimpulan dinamis agar angka selalu berasal dari hasil eksekusi dataset
if not monthly_sales.empty:
    print(
        f"1. Performa penjualan: bulan {best_month['order_month']} memiliki "
        f"pendapatan tertinggi sebesar R$ {best_month['total_revenue']:,.2f}, "
        f"dengan {best_month['total_orders']:,} pesanan."
    )

if not rfm_summary.empty:
    largest_segment = rfm_summary.iloc[0]
    at_risk = rfm_summary[rfm_summary["segment"].eq("At Risk")]
    print(
        f"2. Segmentasi pelanggan: segmen terbesar adalah {largest_segment['segment']} "
        f"({largest_segment['customers']:,} pelanggan; "
        f"{largest_segment['customer_share_pct']:.2f}% dari pelanggan teranalisis)."
    )
    if not at_risk.empty:
        print(f"   Segmen At Risk berjumlah {int(at_risk['customers'].iloc[0]):,} pelanggan.")

if not delivery.empty:
    late_rate = delivery["is_late"].mean() * 100
    avg_days = delivery["delivery_days"].mean()
    print(
        f"3. Pengiriman: tingkat keterlambatan {late_rate:.2f}% dan "
        f"rata-rata durasi pengiriman {avg_days:.2f} hari."
    )

print("\nREKOMENDASI ACTION ITEM:")
print("- Gunakan pola bulan dengan pendapatan tertinggi untuk perencanaan stok dan kapasitas operasional.")
print("- Prioritaskan kampanye win-back untuk segmen At Risk dan program loyalitas untuk Champions.")
print("- Evaluasi mitra logistik/rute dengan tingkat keterlambatan tinggi; tetapkan pemantauan SLA bulanan.")
print("- Pantau metrik penjualan, retensi, dan pengiriman secara rutin melalui dashboard.")

## 9. Export data untuk Dashboard Streamlit

In [ ]:
# Dataset utama dashboard: satu baris per order (hasil join)
dashboard_data = valid_orders[[
    "order_id", "customer_unique_id", "order_purchase_timestamp", "order_month",
    "order_status", "customer_city", "customer_state", "revenue", "freight_value",
    "item_count", "delivery_days", "is_late"
]].copy()

# Tambahkan segmen RFM per pelanggan
dashboard_data = dashboard_data.merge(
    rfm[["customer_unique_id", "segment", "recency", "frequency", "monetary"]],
    on="customer_unique_id", how="left"
)

dashboard_data.to_csv(OUTPUT_DIR / "main_data.csv", index=False)
monthly_sales.to_csv(OUTPUT_DIR / "monthly_sales.csv", index=False)
rfm_summary.to_csv(OUTPUT_DIR / "rfm_segments.csv", index=False)
delivery_by_month.to_csv(OUTPUT_DIR / "delivery_monthly.csv", index=False)

print("File dashboard berhasil diekspor ke folder dashboard/:")
print([p.name for p in OUTPUT_DIR.glob("*.csv")])

## 10. Ringkasan akhir
Notebook ini mencakup seluruh alur analisis data: Gathering, Assessing, Cleaning, EDA, Visualization, Explanatory Analysis, hingga Conclusion & Recommendation. Nilai kesimpulan dihitung otomatis setelah dataset asli dimuat dan seluruh sel dijalankan.

**Sebelum submission:** pastikan seluruh CSV dataset berada di folder `data/`, jalankan ulang notebook dari awal sampai akhir, periksa grafik dan kesimpulan, lalu jalankan dashboard Streamlit.